# Simple linear regression with no tuning

In [69]:
import numpy as np
import pandas as pd
import sys

from pathlib import Path
from sklearn.linear_model import LogisticRegression

# Own functions
# Necessary to import from src dir
sys.path.append("..")
import src.kaggle_helpers as kh
import src.preprocessing_minimal as ppm

## Reading in the data

In [70]:
from sklearn.model_selection import train_test_split

data_dir = Path('data')
output_dir = Path('outputs')

train_df = ppm.preprocess_train_data(data_dir / 'train.csv',drop_std=True)
final_test_df = pd.read_csv(data_dir / 'test.csv').drop(columns=["partlybad", "date"])
final_test_df = final_test_df.drop(columns=[col for col in final_test_df.columns if col.endswith('.std')])

train_df, test_df = train_test_split(train_df, test_size=0.3, random_state=42)

#train_df = ppm.generate_synthetic_data(train_df, num_datasets=100)

## Splitting to x and y variables

In [71]:
X_train, y_train2, y_train4 = ppm.split_xy(train_df)
X_test, y_test2, y_test4 = ppm.split_xy(test_df)
X_test = X_test.drop(columns=[col for col in X_test.columns if col.endswith('.std')])

## Scaling data

In [72]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Constructing event-nonevent classifier

In [73]:
#Fitting logistic regression to class2
log_model = LogisticRegression(C=0.4, l1_ratio=1, solver='liblinear')
log_model.fit(X_train_scaled, y_train2)

from sklearn.metrics import classification_report

y_pred = log_model.predict(X_test_scaled)
print(classification_report(y_test2, y_pred))


              precision    recall  f1-score   support

       event       0.86      0.89      0.87        70
    nonevent       0.87      0.85      0.86        65

    accuracy                           0.87       135
   macro avg       0.87      0.87      0.87       135
weighted avg       0.87      0.87      0.87       135



## Testing different class4 classification methods

In [74]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=160, random_state=42)
rf.fit(X_train_scaled, y_train4)
print(classification_report(y_test4, rf.predict(X_test_scaled)))
print(f"Random Forest Accuracy: {rf.score(X_test_scaled, y_test4):.4f}")


              precision    recall  f1-score   support

          II       0.42      0.54      0.47        35
          Ia       0.50      0.12      0.20         8
          Ib       0.50      0.30      0.37        27
    nonevent       0.79      0.88      0.83        65

    accuracy                           0.63       135
   macro avg       0.55      0.46      0.47       135
weighted avg       0.62      0.63      0.61       135

Random Forest Accuracy: 0.6296


In [75]:
from sklearn.svm import SVC

svm = SVC(kernel="rbf", random_state=42)
svm.fit(X_train_scaled, y_train4.values.ravel())
print(classification_report(y_test4, svm.predict(X_test_scaled)))
print(f"SVM Accuracy: {svm.score(X_test_scaled, y_test4):.4f}")

              precision    recall  f1-score   support

          II       0.38      0.60      0.47        35
          Ia       0.00      0.00      0.00         8
          Ib       0.20      0.07      0.11        27
    nonevent       0.80      0.86      0.83        65

    accuracy                           0.59       135
   macro avg       0.35      0.38      0.35       135
weighted avg       0.52      0.59      0.54       135

SVM Accuracy: 0.5852


/opt/homebrew/anaconda3/envs/ml26/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/anaconda3/envs/ml26/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/anaconda3/envs/ml26/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

### Calculating a estimated Kaggle score

In [76]:
total_score = kh.simulate_kaggle_scoring(X_test_scaled, y_test2, y_test4, log_model, log_model, rf, print_partial_scores=True)
print(f'Kaggle score: {total_score:.4f}')

Binary accuracy: 0.8667
Multiclass accuracy: 0.6296
Model perplexity: 1.3774
Kaggle score: 0.7063


### Forming the output df

In [77]:
out_df = kh.generate_kaggle_output(
    final_test_df,
    model_prob=log_model,
    model_class4=rf,
    scaler=scaler
)
out_df.to_csv(output_dir / 'not_good_submission.csv', index=False)
out_df.head(10)

,id,class4,p
0,450,nonevent,0.642346
1,451,Ib,0.977407
2,452,nonevent,0.017060
3,453,nonevent,0.321294
4,454,Ib,0.862749
5,455,nonevent,0.130364
6,456,Ib,0.913405
7,457,Ib,0.968219
8,458,nonevent,0.266165
9,459,nonevent,0.173111
